<a href="https://colab.research.google.com/github/nikonikoni123/data_analyst_bot/blob/main/program.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Historias de Usuario:

# HU-01: Búsqueda Semántica de Variables

- Como analista de datos que desconoce el esquema exacto de la base de datos quiero poder buscar columnas usando sinónimos o descripciones en lenguaje natural,

- Como analista de datos quiero agilizar el análisis sin tener que imprimir y leer manualmente la lista de columnas (df.columns) constantemente.

# HU-02: Detección Automática de Tipos de Datos (Inferencia)
- Como científico de datos, quiero que el sistema distinga automáticamente si una columna numérica es realmente una categoría o un valor continuo.

- Como científico de datos, Quiero que los gráficos generados sean estadísticamente correctos sin intervención manual.


# HU-03: Explicabilidad Automática (Causalidad)

- Como gerente de negocio, Quiero preguntar "¿Por qué ocurre X?" y obtener una lista de los factores que más influyen en esa variable,
Para tomar decisiones basadas en datos sin esperar a que un equipo de Data Science entrene un modelo manual.

# HU-04: Visualización Inteligente
Como usuario final, Quiero que el sistema elija el mejor gráfico posible (Scatter, Boxplot, Histograma) basándose en los datos que seleccioné,
Para visualizar la información rápidamente sin tener que configurar ejes o tipos de gráficos manualmente.



## Código:

In [ ]:
!pip install -q pandas numpy plotly scikit-learn scipy seaborn xgboost lightgbm catboost deep_translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import re
import math
from typing import List, Dict, Tuple, Optional, Any
import warnings
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# Configuración para Colab
try:
    pio.renderers.default = 'colab'
except Exception:
    pio.renderers.default = 'notebook'

warnings.filterwarnings('ignore')

# Imports de Machine Learning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import r2_score, mean_absolute_error, accuracy_score

# Detección dinámica de librerías de boosting
_available_models = {}
try: import lightgbm as lgb; _available_models['lgbm'] = True
except: _available_models['lgbm'] = False
try: import xgboost as xgb; _available_models['xgboost'] = True
except: _available_models['xgboost'] = False
try: from catboost import CatBoostRegressor, CatBoostClassifier; _available_models['catboost'] = True
except: _available_models['catboost'] = False


def pretty(col: str) -> str:
    """Hace legible el nombre de una columna."""
    if col is None: return ""
    s = str(col).replace('_', ' ').strip()
    s = re.sub(r'\s+', ' ', s)
    return " ".join([w.capitalize() for w in s.split()])

def sample_df(df: pd.DataFrame, n: int = 5000, random_state: int = 42) -> pd.DataFrame:
    """Muestrea el DF para gráficos rápidos si es muy grande."""
    if df is None: return df
    if len(df) > n:
        return df.sample(n=n, random_state=random_state)
    return df

def preprocesar_texto_simple(t: str) -> str:
    """Normalización básica de texto para búsquedas."""
    if t is None: return ''
    t = str(t).lower()
    t = re.sub(r'[^0-9a-zA-Záéíóúüñ\s]', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t

def clasificar_columna_inteligente(s: pd.Series, sample_size: int = 2000) -> str:
    """
    Infiere el tipo de dato real (más allá del dtype de pandas).
    Retorna: 'numerico_continuo', 'numerico_ordinal', 'categoria', 'fecha', 'texto', 'id'
    """
    try:
        if s is None: return 'categoria'
        s_n = s.dropna()
        if len(s_n) == 0: return 'categoria'

        # Muestreo para velocidad
        s_s = s_n.sample(min(len(s_n), sample_size), random_state=42) if len(s_n) > sample_size else s_n

        # 1. Numéricos
        if pd.api.types.is_numeric_dtype(s_s):
            unique_ratio = s_s.nunique() / len(s_s)
            if unique_ratio < 0.05 and s_s.nunique() < 20:
                return 'numerico_ordinal' # Pocos valores, probablemente categorías numéricas (ej. rating 1-5)
            return 'numerico_continuo'

        # 2. Fechas (Heurística)
        if pd.api.types.is_datetime64_any_dtype(s_s):
            return 'fecha'

        # Intentar convertir strings a fechas
        ss = s_s.astype(str)
        if ss.str.match(r'\d{4}-\d{2}-\d{2}').mean() > 0.8:
            return 'fecha'

        # 3. Categorías vs Texto vs ID
        unique_ratio = ss.nunique() / len(ss)
        avg_len = ss.str.len().mean()

        if unique_ratio > 0.95 and avg_len < 20:
            return 'id'
        if unique_ratio < 0.2 or ss.nunique() < 50:
            return 'categoria'
        if avg_len > 50:
            return 'texto'

        return 'categoria' # Fallback seguro
    except Exception:
        return 'categoria'

def perfil_columna(df: pd.DataFrame, col: str) -> Dict[str, Any]:
    """Genera metadatos estadísticos de una columna."""
    s = df[col]
    tp = clasificar_columna_inteligente(s)
    nonnull = s.dropna()
    profile = {
        'col': col,
        'pretty': pretty(col),
        'type': tp,
        'nulls': int(s.isna().sum()),
        'nunique': int(s.nunique())
    }

    try:
        if tp in ('numerico_continuo', 'numerico_ordinal'):
            sx = pd.to_numeric(nonnull, errors='coerce')
            profile.update({
                'mean': float(sx.mean()),
                'median': float(sx.median()),
                'std': float(sx.std()),
                'min': float(sx.min()),
                'max': float(sx.max())
            })
        elif tp == 'categoria':
            profile['top_values'] = nonnull.value_counts().head(5).to_dict()
    except: pass
    return profile


# clase para deteccion de features

class AutoPredictor:
    """Clase auxiliar para ejecutar ML rápido y detectar Feature Importance."""

    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()

    def _preparar_datos(self, target: str) -> Tuple[pd.DataFrame, pd.Series, List[str]]:
        df_clean = self.df.dropna(subset=[target]).copy()
        y = df_clean[target]
        X = df_clean.drop(columns=[target])

        # Selección simple de features
        num_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
        # One-Hot encoding ligero para categorías de baja cardinalidad
        cat_cols = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c]) and X[c].nunique() < 20]

        X = pd.get_dummies(X[num_cols + cat_cols], columns=cat_cols, drop_first=True)
        X = X.fillna(X.median()) # Imputación simple
        return X, y, X.columns.tolist()

    def explicar_variable(self, target: str) -> Dict[str, Any]:
        """Entrena un modelo rápido para decir qué variables influyen en el target."""
        if target not in self.df.columns:
            return {"error": f"Columna {target} no encontrada."}

        try:
            X, y, features = self._preparar_datos(target)
            if X.empty or len(X) < 50: return {"error": "Datos insuficientes para ML."}

            is_reg = pd.api.types.is_numeric_dtype(y) and y.nunique() > 20

            # Selección de modelo robusto
            model = None
            if is_reg:
                if _available_models['xgboost']: model = xgb.XGBRegressor(verbosity=0)
                else: model = LinearRegression()
            else:
                # Si es clasificación, asegurar labels numéricos
                y = pd.factorize(y)[0]
                if _available_models['xgboost']: model = xgb.XGBClassifier(verbosity=0, use_label_encoder=False)
                else: model = LogisticRegression(max_iter=1000)

            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
            model.fit(X_train, y_train)

            score = model.score(X_test, y_test)

            # Feature Importance
            imps = []
            if hasattr(model, 'feature_importances_'):
                imps = zip(features, model.feature_importances_)
            elif hasattr(model, 'coef_'):
                imps = zip(features, abs(model.coef_[0]) if hasattr(model.coef_, '__iter__') and len(model.coef_.shape)>1 else abs(model.coef_))

            imps = sorted(imps, key=lambda x: x[1], reverse=True)[:10]

            return {
                "type": "regression" if is_reg else "classification",
                "score": score,
                "metric": "R2" if is_reg else "Accuracy",
                "importances": imps
            }
        except Exception as e:
            return {"error": str(e)}

# MÓDULO AVANZADO DE VISUALIZACIÓN

class VisualizerEngine:
    """
    Motor especializado en decidir y renderizar el gráfico correcto
    basándose en reglas de negocio estrictas y topología de datos.
    """

    @staticmethod
    def validar_datos(df: pd.DataFrame, cols: List[str]) -> bool:
        """Reglas de rechazo: Si los datos no sirven, no graficamos."""
        if df.empty: return False
        for c in cols:
            if c not in df.columns: return False
            if df[c].isna().all(): return False
            # Valor constante (sin varianza)
            if df[c].nunique() <= 1: return False
        return True

    @staticmethod
    def generar_heatmap_correlacion(df: pd.DataFrame, cols: List[str]) -> Optional[go.Figure]:
        """Genera matriz de correlación si hay múltiples numéricas."""
        num_cols = [c for c in cols if pd.api.types.is_numeric_dtype(df[c])]
        # Necesitamos al menos 3 para que valga la pena un heatmap
        if len(num_cols) < 3: return None

        corr = df[num_cols].corr()
        fig = px.imshow(corr, text_auto=".2f", aspect="auto", color_continuous_scale='RdBu_r', title="Matriz de Correlación")
        return fig

    @staticmethod
    def generar_crosstab(df: pd.DataFrame, col_a: str, col_b: str) -> go.Figure:
        """Mapa de calor para cruce de dos categorías (Densidad)."""
        crosstab = pd.crosstab(df[col_a], df[col_b])
        fig = px.imshow(crosstab, text_auto=True, title=f"Densidad: {pretty(col_a)} vs {pretty(col_b)}")
        return fig

    @staticmethod
    def generar_scatter_avanzado(df: pd.DataFrame, x: str, y: str, color_col: Optional[str] = None) -> go.Figure:
        """Scatter plot con línea de tendencia y dimensión de color opcional."""
        title = f"Correlación: {pretty(y)} vs {pretty(x)}"
        if color_col:
            title += f" (por {pretty(color_col)})"

        fig = px.scatter(
            df, x=x, y=y,
            color=color_col, # Tercera dimensión
            trendline="ols" if df[x].count() > 10 and df[y].count() > 10 else None,
            title=title,
            opacity=0.7
        )
        return fig

    @staticmethod
    def generar_distribucion_comparada(df: pd.DataFrame, num: str, cat: str) -> go.Figure:
        """Violin plot: Mejor que boxplot para ver densidad y outliers."""
        fig = px.violin(
            df, x=cat, y=num,
            box=True, # Muestra el boxplot adentro
            points="all", # Muestra los puntos (dispersión)
            hover_data=df.columns,
            title=f"Distribución de {pretty(num)} por {pretty(cat)}"
        )
        return fig

    @staticmethod
    def generar_serie_tiempo(df: pd.DataFrame, date_col: str, num_col: str) -> go.Figure:
        """Línea evolutiva agregada por tiempo."""
        # Agregamos por fecha para limpiar ruido
        df_agg = df.groupby(date_col)[num_col].mean().reset_index()
        fig = px.line(df_agg, x=date_col, y=num_col, title=f"Evolución de {pretty(num_col)} en el tiempo")
        fig.update_traces(mode="lines+markers")
        return fig

# CLASE PRINCIPAL

class DataAnalystBot:
    def __init__(self, df: pd.DataFrame):
        if df is None or df.empty:
            raise ValueError("El DataFrame está vacío.")
        self.df = df
        self.profiles = {c: perfil_columna(df, c) for c in df.columns}
        self.predictor = AutoPredictor(df)
        self._preparar_motor_busqueda()

    def _preparar_motor_busqueda(self):
        """Indexación TF-IDF (Igual que antes)"""
        docs = []
        self.col_names = list(self.df.columns)
        for c in self.col_names:
            p = self.profiles[c]
            doc = f"{pretty(c)} {c} tipo_{p['type']}"
            if 'top_values' in p:
                doc += " " + " ".join([str(k) for k in p['top_values'].keys()])
            docs.append(doc)
        self.vectorizer = TfidfVectorizer()
        self.tfidf_matrix = self.vectorizer.fit_transform(docs)

    def _seleccionar_columnas_relevantes(self, query: str, top_k: int = 4) -> List[str]:
        """Búsqueda semántica (Igual que antes)"""
        if not query: return self.col_names[:top_k]
        query_vec = self.vectorizer.transform([preprocesar_texto_simple(query)])
        sims = cosine_similarity(query_vec, self.tfidf_matrix).flatten()
        indices = np.argsort(sims)[::-1]
        relevant_cols = [self.col_names[i] for i in indices if sims[i] > 0]
        return relevant_cols[:top_k] if relevant_cols else self.col_names[:top_k]

    def _decidir_y_graficar(self, cols_seleccionadas: List[str]) -> List[go.Figure]:
        """
        CEREBRO DE VISUALIZACIÓN:
        Decide qué gráfica hacer basándose en la combinación exacta de tipos de datos.
        """
        graficos = []
        df_safe = sample_df(self.df, n=2000) # Trabajamos con muestra segura

        # Filtramos columnas vacías o constantes
        cols_validas = [c for c in cols_seleccionadas if VisualizerEngine.validar_datos(df_safe, [c])]
        if not cols_validas: return []

        # 1. INTENTO DE HEATMAP (Si hay muchas numéricas)
        if len(cols_validas) >= 3:
            fig_corr = VisualizerEngine.generar_heatmap_correlacion(df_safe, cols_validas)
            if fig_corr: graficos.append(fig_corr)

        # 2. ANÁLISIS CRUZADO (PRIORIDAD ALTA)
        # Tomamos las 2 primeras columnas más relevantes
        if len(cols_validas) >= 2:
            c1, c2 = cols_validas[0], cols_validas[1]
            t1, t2 = self.profiles[c1]['type'], self.profiles[c2]['type']

            # Caso A: Numérico vs Numérico (Scatter)
            if t1 in ['numerico_continuo', 'numerico_ordinal'] and t2 in ['numerico_continuo', 'numerico_ordinal']:
                # Intentar buscar una 3ra columna categórica para el color
                col_color = None
                for c_extra in cols_validas[2:]:
                    if self.profiles[c_extra]['type'] == 'categoria' and df_safe[c_extra].nunique() < 10:
                        col_color = c_extra
                        break
                graficos.append(VisualizerEngine.generar_scatter_avanzado(df_safe, c1, c2, col_color))

            # Caso B: Numérico vs Fecha (Serie de Tiempo)
            elif 'fecha' in [t1, t2] and ('numerico_continuo' in [t1, t2] or 'numerico_ordinal' in [t1, t2]):
                date_c = c1 if t1 == 'fecha' else c2
                num_c = c2 if t1 == 'fecha' else c1
                graficos.append(VisualizerEngine.generar_serie_tiempo(self.df, date_c, num_c)) # Usar DF completo para series de tiempo

            # Caso C: Categórico vs Numérico (Violin Plot)
            elif (t1 == 'categoria' and t2.startswith('numerico')) or (t2 == 'categoria' and t1.startswith('numerico')):
                cat_c = c1 if t1 == 'categoria' else c2
                num_c = c2 if t1 == 'categoria' else c1
                # Solo si no son demasiadas categorías
                if df_safe[cat_c].nunique() <= 20:
                    graficos.append(VisualizerEngine.generar_distribucion_comparada(df_safe, num_c, cat_c))

            # Caso D: Categórico vs Categórico (Crosstab Heatmap)
            elif t1 == 'categoria' and t2 == 'categoria':
                if df_safe[c1].nunique() < 30 and df_safe[c2].nunique() < 30:
                    graficos.append(VisualizerEngine.generar_crosstab(df_safe, c1, c2))

        # 3. ANÁLISIS INDIVIDUAL (Univariado)
        # Solo graficamos la columna principal si no se generaron cruces o para reforzar
        primary_col = cols_validas[0]
        ptype = self.profiles[primary_col]['type']

        if ptype.startswith('numerico'):
            graficos.append(px.histogram(df_safe, x=primary_col, marginal="box", title=f"Distribución Univariada: {pretty(primary_col)}"))
        elif ptype == 'categoria' and df_safe[primary_col].nunique() < 50:
            counts = df_safe[primary_col].value_counts().reset_index()
            counts.columns = [primary_col, 'Frecuencia']
            graficos.append(px.bar(counts.head(15), x=primary_col, y='Frecuencia', color='Frecuencia', title=f"Top Categorías: {pretty(primary_col)}"))

        # Ajuste final de diseño
        for fig in graficos:
            fig.update_layout(template="plotly_white", height=450, margin=dict(t=50, b=20, l=20, r=20))

        return graficos

    def analizar(self, peticion: str, top_k: int = 5):
        """Flujo principal de análisis"""
        print(f" Analizando petición: '{peticion}'...")

        # 1. Selección de Columnas
        cols_candidatas = self._seleccionar_columnas_relevantes(peticion, top_k=top_k)
        print(f" Variables identificadas: {', '.join([pretty(c) for c in cols_candidatas])}")

        reporte = []
        graficos = []

        # 2. Machine Learning (Si es pregunta causal)
        if any(kw in peticion.lower() for kw in ['por qué', 'influye', 'impacta', 'causa']):
            target = cols_candidatas[0]
            print(f" Ejecutando análisis causal para: {target}")
            res_ml = self.predictor.explicar_variable(target)
            if "error" not in res_ml:
                reporte.append(f"###  Drivers de '{pretty(target)}'")
                reporte.append(f"Capacidad explicativa del modelo: **{res_ml['score']:.2f}** ({res_ml['metric']})")
                df_imp = pd.DataFrame(res_ml['importances'], columns=['Variable', 'Impacto'])
                graficos.append(px.bar(df_imp.head(10), x='Impacto', y='Variable', orientation='h', color='Impacto', title=f"Impacto de variables en {pretty(target)}"))
            else:
                reporte.append(f" No se pudo generar modelo ML: {res_ml['error']}")

        # 3. Generación Inteligente de Gráficos
        graficos_exploratorios = self._decidir_y_graficar(cols_candidatas)
        graficos.extend(graficos_exploratorios)

        # 4. Renderizado
        print("\n" + "="*60)
        if reporte: print("\n".join(reporte))
        else: print("Generando visualizaciones basadas en la relación de los datos...")
        print("="*60 + "\n")

        if not graficos:
            print(" No se encontraron relaciones gráficas válidas o los datos son constantes/nulos.")

        for g in graficos:
            g.show()

# ejemplo de uso

In [ ]:
# Cargamos dataset Titanic
df = sns.load_dataset('titanic')

# Instanciamos el nuevo bot mejorado
bot = DataAnalystBot(df)

# Prueba 1: Relación compleja (Numérico vs Numérico + Categoría oculta)
# Esto debería generar un Scatter Plot coloreado si encuentra una categórica relevante
bot.analizar("relación entre edad y tarifa")

# Prueba 2: Distribución comparada (Categoría vs Numérico)
# Esto debería generar un Violin Plot
bot.analizar("distribución de edad según la clase en que viajaban")

# Prueba 3: Correlaciones múltiples
# Esto debería generar un Heatmap
bot.analizar("analiza edad, tarifa, hermanos y padres", top_k=5)

 Analizando petición: 'relación entre edad y tarifa'...
 Variables identificadas: Survived, Pclass, Sex, Age, Sibsp

Generando visualizaciones basadas en la relación de los datos...



 Analizando petición: 'distribución de edad según la clase en que viajaban'...
 Variables identificadas: Survived, Pclass, Sex, Age, Sibsp

Generando visualizaciones basadas en la relación de los datos...



 Analizando petición: 'analiza edad, tarifa, hermanos y padres'...
 Variables identificadas: Survived, Pclass, Sex, Age, Sibsp

Generando visualizaciones basadas en la relación de los datos...

